In [ ]:
# Run this cell if using Google Colab or if pyapprox is not installed locally
try:
    import pyapprox
except ImportError:
    !pip install "pyapprox[all]" -q
    # If the PyPI install fails, uncomment the line below to install from source:
    # !pip install "pyapprox[all] @ git+https://github.com/sandialabs/pyapprox.git" -q

---
title: "From Models to Decisions Under Uncertainty"
subtitle: "PyApprox Tutorial Library"
description: "Opening tutorial: why computational models need uncertainty quantification, grounded in a composite cantilever beam."
tutorial_type: concept
topic: uncertainty_quantification
difficulty: beginner
estimated_time: 15
render_time: 13
prerequisites:
  - setup_environment
tags:
  - uq
  - beam
  - concepts
  - forward-uq
format:
  html:
    code-fold: false
    code-tools: true
    toc: true
execute:
  echo: true
  warning: false
jupyter: python3
---



## Learning Objectives

After completing this tutorial, you will be able to:

- Explain why computational models are used to predict complex physical phenomena
- Define a Quantity of Interest (QoI) and explain why predictions are reduced to scalar outputs for decision making
- Identify sources of uncertainty in a computational model and explain why they matter

## Prerequisites

Complete [Setting Up Your Environment](setup_environment.ipynb) before this tutorial.


## Models Predict Complex Physical Phenomena
Engineers and scientists routinely use computational models to predict the behavior of systems that are too complex, too expensive, or too dangerous to test exhaustively in the lab. These models underpin critical decisions---from certifying that a structure is safe, to optimizing the design of a new one. But a prediction is only useful if we understand how much to trust it. Uncertainty quantification (UQ) provides the mathematical framework for answering that question.
This tutorial is the first in a series introducing core UQ concepts and the PyApprox tools that implement them. We begin with a concrete model problem: a composite cantilever beam with stiff outer skins, a compliant inner core containing circular holes, clamped at one end and loaded on top (@fig-beam-geometry).

In [ ]:
#| echo: false
#| fig-cap: "Composite cantilever beam: three material layers with five circular holes. Clamped at the left edge, linearly increasing traction on the top surface."
#| label: fig-beam-geometry
import numpy as np
np.random.seed(42)
import matplotlib.pyplot as plt
from _figures import plot_mdu_beam_geometry

fig, ax = plt.subplots(figsize=(10, 4))
plot_mdu_beam_geometry(ax)
plt.tight_layout()
plt.show()

The beam has three subdomains: stiff skins on the top and bottom ($E_1, \nu_1$) and a compliant core with circular holes of radius $r$ ($E_2, \nu_2$). The governing physics is linear elasticity. In each material layer, the stress-strain relationship follows Hooke's law:

$$
\sigma_{ij} = \lambda \, \varepsilon_{kk} \, \delta_{ij} + 2\mu \, \varepsilon_{ij}
$$ {#eq-hooke}

where the Lamé parameters $\lambda$ and $\mu$ depend on the layer's Young's modulus $E_i$ and Poisson ratio $\nu_i$. Equilibrium and boundary conditions complete the boundary value problem:

$$
\nabla \cdot \boldsymbol{\sigma} = \mathbf{0} \;\text{in}\; \Omega, \qquad
\mathbf{u} = \mathbf{0} \;\text{on}\; \Gamma_D, \qquad
\boldsymbol{\sigma} \cdot \mathbf{n} = \mathbf{t} \;\text{on}\; \Gamma_N
$$ {#eq-strong-form}

where $\mathbf{t} = (0,\, -q_0 x/L)^T$ is the linearly increasing traction on the top surface.

Solving this PDE with a finite element method produces a rich, high-dimensional output: displacement and stress fields over the entire domain. @fig-reference-solution shows the result at fixed (nominal) material properties.

In [ ]:
#| echo: false
#| fig-cap: "Von Mises stress on the deformed mesh (magnified) with undeformed outline. Stress concentrations appear at the holes near the clamped end."
#| label: fig-reference-solution
from _figures import plot_reference_solution

fig, ax = plt.subplots(figsize=(10, 4))
plot_reference_solution(fig, ax)
plt.tight_layout()
plt.show()

## From Full Solutions to Quantities of Interest

The simulation in @fig-reference-solution produces displacement and stress values at thousands of mesh nodes. But an engineer rarely needs the entire field. Instead, they need to answer specific questions that drive decisions:

- **Does the beam deflect too much?** → Measure the vertical tip displacement $\delta_{\text{tip}} = u_y(L, H/2)$.
- **Will the material yield?** → Check the total von Mises stress integrated over the domain, $\sigma_{\text{tot}} = \int_\Omega \sigma_{\text{vm}}\, dA$.

Each of these is a **Quantity of Interest (QoI)**: a scalar (or low-dimensional) functional of the full model solution that directly informs a decision. In UQ, we focus on characterizing uncertainty in the QoI, not the entire solution field.

The PyApprox beam benchmark returns both QoIs:

$$
q_1 = \delta_{\text{tip}} = u_y(L,\, H/2), \qquad q_2 = \sigma_{\text{tot}} = \int_\Omega \sigma_{\text{vm}}\, dA
$$ {#eq-qoi}

Throughout these tutorials, we use $\delta_{\text{tip}}$ as the primary QoI for uncertainty propagation and $\sigma_{\text{tot}}$ as a secondary QoI that captures global stress information.


## Models Are Subject to Uncertainties

So far, we solved the beam at fixed, nominal material properties. But in practice, the inputs to a model are **uncertain**:

- **Parameter uncertainty.** The Young's moduli ($E_1$, $E_2$) and Poisson ratios ($\nu_1$, $\nu_2$) are not known exactly. Manufacturing tolerances, batch-to-batch variability, and limited test data all introduce uncertainty. @fig-parameter-uncertainty shows how changing the material stiffness changes the beam's response: halving the moduli roughly doubles the deflection.

In [ ]:
#| echo: false
#| fig-cap: "Effect of parameter uncertainty. Left: stiff materials ($E_1 = 2.5{\\times}10^4$, $E_2 = 6{\\times}10^3$). Right: soft materials ($E_1 = 1.2{\\times}10^4$, $E_2 = 3{\\times}10^3$). The same load produces roughly twice the deflection in the softer beam."
#| label: fig-parameter-uncertainty
from _figures import plot_uncertainty_sources

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5),
                               gridspec_kw={"right": 0.88})
plot_uncertainty_sources("parameter", fig, (ax1, ax2))
plt.show()

- **Model-form uncertainty.** The governing equations themselves are approximations. Linear elasticity (@eq-hooke) assumes small deformations, but under heavy loading the geometry changes enough to stiffen the response. A **Neo-Hookean** hyperelastic model captures this geometric nonlinearity. @fig-model-form-uncertainty shows both models at the same (increased) load: the linear model over-predicts deflection by about 58% because it ignores the stiffening effect of large rotations.

In [ ]:
#| echo: false
#| fig-cap: "Model-form uncertainty. Left: linear elasticity. Right: Neo-Hookean hyperelasticity. At higher load ($q_0 = 120$), the linear model over-predicts deflection by 58% because it neglects geometric stiffening."
#| label: fig-model-form-uncertainty

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5),
                               gridspec_kw={"right": 0.88})
plot_uncertainty_sources("model_form", fig, (ax1, ax2))
plt.show()

- **Numerical uncertainty.** The finite element solution depends on the mesh: a coarser mesh is cheaper but less accurate. @fig-numerical-uncertainty compares solutions on two meshes; the coarse mesh under-predicts tip deflection by about 6%.

In [ ]:
#| echo: false
#| fig-cap: "Numerical uncertainty. Left: coarse mesh ($h = 4$, 199 nodes). Right: fine mesh ($h = 2$, 746 nodes). The coarse mesh under-predicts tip deflection by about 6%."
#| label: fig-numerical-uncertainty

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5),
                               gridspec_kw={"right": 0.88})
plot_uncertainty_sources("numerical", fig, (ax1, ax2))
plt.show()

- **Data uncertainty.** If we calibrate the model against experimental measurements (e.g., measured tip deflection from a lab test), those measurements carry their own noise and systematic errors.



## Quantifying the Impact of Uncertainty on the QoI

::: {.callout-important}
## The central challenge
A single model run at nominal inputs gives a single number for $\delta_{\text{tip}}$. But the true value of $\delta_{\text{tip}}$ is not a single number---it is a **distribution** determined by the model uncertainties. Understanding that distribution is the purpose of UQ.
:::
	

The preceding examples made one thing clear: the QoI is not a single number. Parameter values we cannot pin down exactly, modeling assumptions we know are imperfect, and meshes we know are finite all shift #\delta_{\text{tip}}$ — sometimes by a few percent, sometimes by more than 50%. And these sources of uncertainty do not act in isolation; in a real analysis they compound.

A prediction that ignores this variability is, at best, incomplete — and at worst, dangerously misleading for the decisions it is meant to support. Before we can certify a safety margin, optimize a design, or calibrate against experimental data, we need rigorous answers to questions like: How much can the QoI change given what we do and do not know? Which uncertainties dominate? How confident should we be in the prediction?

## Key Takeaways

- Computational models predict complex physical behavior, but decisions are based on specific Quantities of Interest — scalar outputs like tip deflection or integrated stress
- Model predictions are subject to multiple sources of uncertainty: uncertain material parameters, imperfect modeling assumptions, finite mesh resolution, and noisy data
- Each source can significantly change the QoI — parameter variation doubled the deflection, and linear vs. nonlinear constitutive models disagreed by over 50%
- A single prediction at nominal inputs, without accounting for these uncertainties, is insufficient for reliable decision making
- Uncertainty quantification provides the tools to characterize, propagate, and manage these uncertainties — the remaining tutorials introduce those tools

## Next Steps

Continue with:

- [Random Variables & Distributions](random_variables_distributions.ipynb) -- Specifying parameter uncertainty mathematically